# ICS 604: APPLIED DATA SCIENCE

## Data Wrangling: Joining, Combining and Reshaping Data 

---

In [ ]:
import numpy as np
import pandas as pd

## Combining and Merging Datasets

Pandas provides several ways to combine and merge DataFrames, allowing you to bring together data from multiple sources. These operations are conceptually similar to combining tables in relational databases and offer flexibility depending on how the datasets are related.

Two core operations are commonly used. The `merge()` function connects rows from different DataFrames based on one or more key columns, closely resembling SQL-style join operations. The `concat()` function, on the other hand, is used to combine DataFrames by appending them either vertically (stacking rows) or horizontally (adding columns). Together, these methods cover most use cases for assembling datasets in pandas.

In [ ]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'a', 'b'], 'data1': range(7)})
df1

In [ ]:
df2 = pd.DataFrame({'key': ['a', 'b', 'd'], 'data2': range(3)})
df2

### Database-Style DataFrame Joins

andas supports database-style joins that allow you to merge DataFrames based on common keys, similar to SQL join operations. In the example shown, both `df1` and `df2` contain a column named `key`, which can be used to align rows from the two DataFrames.

You can merge `df1` and `df2` on the shared column `key`. Pandas can automatically infer the overlapping column name, but it is often safer to specify it explicitly using the `on=` parameter — especially in large datasets with many columns — to avoid unintended joins. In this case, `df1` contains multiple rows with the same `key` value, while `df2` has only one row per `key`. As a result, the merge operation corresponds to a **many-to-one** join, where multiple rows from `df1` are matched to a single row in `df2`.

In [ ]:
merged_df = pd.merge(df1, df2)
merged_df

In [ ]:
merged_df = pd.merge(df1, df2, on='key')
merged_df

In [ ]:
merged_df.sort_values("key")

If the join columns have different names in the two DataFrames, you can specify them explicitly using the `left_on=` and `right_on=` parameters. These arguments tell pandas which column in the left DataFrame and which column in the right DataFrame should be used as the join keys, ensuring the correct alignment of rows even when column names do not match.

In [ ]:
df3 = pd.DataFrame({'lkey': ['b', 'b', 'a', 'c', 'a', 'a', 'b'], 'data1': range(7)})
df4 = pd.DataFrame({'rkey': ['a', 'b', 'd'], 'data2': range(3)})
display(df3)
display(df4)
merged_df = pd.merge(df3, df4, left_on='lkey', right_on='rkey')
merged_df[["lkey", "rkey", "data1", "data2"]]

In [ ]:
merged_df

In the example shown, some key values appear in only one of the two DataFrames—specifically, `lkey = 'c'` exists only in the left DataFrame and `rkey = 'd'` exists only in the right DataFrame. These values do not appear in the merged result.

### Merge Types

By default, `pandas.merge()` performs an **inner join**, meaning that only rows with keys present in both DataFrames are included in the result. Rows with unmatched keys are dropped.

Pandas supports several merge types that control how keys from the two DataFrames are combined:

| Option  | Behavior      |
| ------- |:------------- | 
| `inner` | Keeps only keys present in both DataFrames (intersection) | 
| `left`  | Keeps all keys from the left DataFrame | 
| `right` | Keeps all keys from the right DataFrame | 
| `outer` | Keeps all keys from both DataFrame (union) | 
| `cross` | Returns the Cartesian product of the two DataFrames (cross join) |

In [ ]:
pd.merge(df1, df2, on='key', how='outer')

In [ ]:
pd.merge(df3, df4, left_on='lkey', right_on='rkey', how='outer')

In [ ]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'b'],
                    'data1': range(6)})
df1

In [ ]:
df2 = pd.DataFrame({'key': ['a', 'b', 'a', 'b', 'd'],
                    'data2': range(5)})
df2

In [ ]:
pd.merge(df1, df2, on='key', how='left')

In [ ]:
pd.merge(df1, df2, how='right')

In [ ]:
pd.merge(df1, df2, how='inner')

In [ ]:
pd.merge(df1, df2)   # inner join by default

In [ ]:
pd.merge(df1, df2, how='cross')

In [ ]:
left = pd.DataFrame({'key1': ['foo', 'foo', 'bar'],
                     'key2': ['one', 'two', 'one'],
                     'lval': [1, 2, 3]})
left

In [ ]:
right = pd.DataFrame({'key1': ['foo', 'foo', 'bar', 'bar'],
                      'key2': ['one', 'one', 'one', 'two'],
                      'rval': [4, 5, 6, 7]})
right

In [ ]:
# merge with multiple keys

pd.merge(left, right, on=['key1', 'key2'], how='outer')

### Avoid Column Collision

When merging DataFrames, it is common for both tables to contain columns with the same name. To avoid ambiguity in the resulting DataFrame, pandas automatically **renames overlapping columns**.

By default, when `merge()` encounters identical column names that are not used as join keys, pandas appends the suffix `_x` to columns from the left DataFrame and `_y` to columns from the right DataFrame. This behavior ensures that no column names collide and that the origin of each column remains clear after the merge.

In [ ]:
pd.merge(left, right, on='key1')

In [ ]:
pd.merge(left, right, on='key1', suffixes=('_left', '_right'))

### Merging on Index

In addition to merging on columns, pandas also allows you to merge DataFrames using their indexes, which is a very common operation. This is especially useful when the index already represents a meaningful key, such as an identifier or a timestamp.

Merging on one or more index levels — including hierarchical (MultiIndex) keys — works the same way as merging on multiple columns. The same join types (`inner`, `left`, `right`, `outer`) apply, and pandas aligns the DataFrames based on matching index values.

In [ ]:
left1 = pd.DataFrame({'key': ['a', 'b', 'a', 'a', 'b', 'c'],
                      'value': range(6)})
left1

In [ ]:
right1 = pd.DataFrame({'group_val': [3.5, 7]}, index=['a', 'b'])
right1

In [ ]:
pd.merge(left1, right1, left_on='key', right_index=True)

In [ ]:
pd.merge(left1, right1, left_on='key', right_index=True, how='outer')

In [ ]:
left_hierarchical = pd.DataFrame({'key1': ['Ohio', 'Ohio', 'Ohio', 'Nevada', 'Nevada'],
                                 'key2': [2000, 2001, 2002, 2001, 2002],
                                 'data': np.arange(5.)})
left_hierarchical

In [ ]:
right_hierarchical = pd.DataFrame(np.arange(12).reshape((6, 2)),
                                 index=[['Nevada', 'Nevada', 'Ohio', 'Ohio','Ohio', 'Ohio'],
                                        [2001, 2000, 2000, 2000, 2001, 2002]],
                                 columns=['event1', 'event2'])
right_hierarchical

In [ ]:
pd.merge(left_hierarchical, right_hierarchical, 
         left_on=['key1', 'key2'], right_index=True)

In [ ]:
pd.merge(left_hierarchical, right_hierarchical, 
         left_on=['key1', 'key2'], right_index=True, how='outer')

In [ ]:
# reset index after merge to avoid confusion with duplicate indices
pd.merge(left_hierarchical, right_hierarchical, 
         left_on=['key1', 'key2'], right_index=True, how='outer').reset_index(drop=True)

In [ ]:
left2 = pd.DataFrame([[1., 2.], [3., 4.], [5., 6.]],
                     index=['a', 'c', 'e'],
                     columns=['Ohio', 'Nevada'])
left2

In [ ]:
right2 = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [13, 14]],
                      index=['b', 'c', 'd', 'e'],
                      columns=['Missouri', 'Alabama'])
right2

In [ ]:
pd.merge(left2, right2, how='outer', left_index=True, right_index=True)

## Concatenating Along an Axis

Concatenation, also known as binding or stacking, is another way to combine pandas objects. It is used to place DataFrames or Series together along a specified axis rather than matching rows based on keys.

By default, concatenation operates along `axis=0`, meaning objects are stacked vertically by adding rows. It also uses `join='outer'`, which means the union of columns is kept, and missing values are filled with NaN where a column does not exist in one of the objects.


In [ ]:
arr = np.arange(12).reshape((3, 4))
arr

In [ ]:
np.concatenate([arr, arr])

In [ ]:
np.concatenate([arr, arr], axis=1)

In [ ]:
s1 = pd.Series([0, 1], index=['a', 'b'])
s2 = pd.Series([2, 3, 4], index=['c', 'd', 'e'])
s3 = pd.Series([5, 6], index=['f', 'g'])

print(s1)

print("*" * 10)
print(s2)

print("*" * 10)
print(s3)

In [ ]:
pd.concat([s1, s2, s3])

In [ ]:
# Each Series becomes a separate column.
# Since the Series have no names, default column labels 0, 1, 2 are assigned.
# The row index is the union of all indices (a–g), with missing values filled with NaN.
pd.concat([s1, s2, s3], axis=1)

In [ ]:
# set column labels using the 'keys' parameter
pd.concat([s1, s2, s3], axis=1, keys=['s1', 's2', 's3'])

In [ ]:
print(s1)
print("*" * 10)
print(s3)

s4 = pd.concat([s1, s3])
s4

In [ ]:
pd.concat([s1, s4], axis=1)

In [ ]:
pd.concat([s1, s4], axis=1, join='inner')

### Creating Indexes by Concatenation

Concatenation can also be used to create a hierarchical (MultiIndex) structure in the resulting object. By supplying the `keys=` parameter — one key for each object being concatenated — pandas adds an additional level to the index that identifies the source of each piece of data.

When concatenating along `axis=1`, the behavior depends on the object type. For Series, the keys become the column names in the resulting DataFrame. For DataFrames, the keys form a new level in the column index, creating hierarchical column labels that indicate where each set of columns originated.

In [ ]:
display(s1)
display(s2)
display(s3)

In [ ]:
result = pd.concat([s1, s2, s3], 
                   keys=['patient_1', 'patient_2', 'patient_3'])
result

In [ ]:
result.index

In [ ]:
result.unstack()

In [ ]:
pd.concat([s1, s2, s3], axis=1, keys=['COL_1', 'COL_2', 'COL_3'])

In [ ]:
df1 = pd.DataFrame(np.arange(6).reshape(3, 2), index=['a', 'b', 'c'], columns=['one', 'two'])
df2 = pd.DataFrame(5 + np.arange(4).reshape(2, 2), index=['a', 'c'], columns=['three', 'four'])

display(df1)
display(df2)

In [ ]:
pd.concat([df1, df2], axis=1, keys=['level1', 'level2'])

In [ ]:
pd.concat({'level1': df1, 'level2': df2}, axis=1)

In [ ]:
pd.concat([df1, df2], axis=1, keys=['level1', 'level2']).columns

In [ ]:
df1 = pd.DataFrame(np.random.randn(3, 4), columns=['a', 'b', 'c', 'd'])
df2 = pd.DataFrame(np.random.randn(2, 3), columns=['b', 'd', 'a'])

display(df1, df2)

In [ ]:
pd.concat([df1, df2])

In [ ]:
pd.concat([df1, df2], ignore_index=True)

In [ ]:
pd.concat([df1, df2], axis=1)

In [ ]:
pd.concat([df1, df2], join='inner')

In [ ]:
pd.concat([df1, df2], axis=1, join='inner')

For a more detailed list of options and arguments available in `concat()`, see Table 8.3 in *Python for Data Analysis*.

### Combining Data with Overlap

The `combine_first()` method provides a convenient way to **merge two pandas objects while preserving data from one as a priority**. It works similarly to a conditional operation: for each element, if the value in the first object is not missing, it is retained; otherwise, the corresponding value from the second object is used.

This is particularly useful when combining datasets that partially overlap or when filling missing values in one DataFrame with corresponding values from another.

In [ ]:
s1 = pd.Series([np.nan, 2.5, np.nan, 3.5, 4.5, np.nan],
              index=['f', 'e', 'd', 'c', 'b', 'a'])
s2 = pd.Series(np.arange(len(s1), dtype=np.float64),
              index=['a', 'b', 'c', 'd', 'e', 'f'])
s2.iloc[-1] = np.nan

display(s1)
display(s2)

In [ ]:
# numpy.where() ignores index alignment
np.where(pd.isna(s1), s2, s1)

Unlike `numpy.where()`, which ignores index alignment and does not require the objects to have the same length, the pandas `combine_first()` method aligns values by index. This ensures that values from the first Series are retained when present, and missing values are filled from the second Series according to matching index labels.

In [ ]:
# combine_first() aligns index values
s1.combine_first(s2)

With DataFrame objects, `combine_first()` operates *column by column*, effectively “patching” missing values in the calling DataFrame with values from the DataFrame passed as an argument.

The resulting DataFrame will include the *union of all column names* from both objects, ensuring that no data is lost even if one DataFrame contains columns the other does not.

In [ ]:
df1 = pd.DataFrame({'a': [1., np.nan, 5., np.nan], 
                    'b': [np.nan, 2., np.nan, 6.],
                    'c': range(2, 18, 4)})
df2 = pd.DataFrame({'a': [5., 4., np.nan, 3., 7.], 
                    'b': [np.nan, 3., 4., 6., 8.]})

display(df1)
display(df2)
df1.combine_first(df2)